# Answers: 5 · Classes and call flow

Worked answers to the **Exercises** at the end of
[`classes_and_call_flow.ipynb`](../../00_how_al_molecular_works/classes_and_call_flow.ipynb).
Try each one yourself first. That lesson uses a small **fake** pool (3,000 molecules), so these
answers do too. The first code cell recreates it.

In [ ]:
import sys, io, contextlib, tempfile, traceback, inspect
from pathlib import Path
import numpy as np, torch
sys.path.insert(0, "/N/slate/mengjing/repos/al-molecular-tutorial")
sys.path.insert(0, "../../00_how_al_molecular_works")       # for calltrace.py
import run_experiment as rx
from calltrace import trace

torch.manual_seed(0)
rng = np.random.default_rng(0)
N = 3000
pool_smiles = [f"mol_{i}" for i in range(N)]
hidden = rng.normal(size=(N, 4)).astype("float32")
emb_dict = {
    "molformer": np.c_[hidden, rng.normal(size=(N, 12))].astype("float32"),
    "grover":    (hidden @ rng.normal(size=(4, 8))).astype("float32"),
}
true_scores = -8 - (hidden[:, 0] + 0.5 * hidden[:, 1] + 0.3 * hidden[:, 2] * hidden[:, 3])
oracle = dict(zip(pool_smiles, true_scores.astype(float)))

def explorer(cls=rx.MVEExplorer, n_rounds=2, surrogate_type="single", **kw):
    return cls(emb_dict=emb_dict, pool_smiles=pool_smiles, oracle=oracle,
               surrogate_type=surrogate_type, backbone="molformer", acq="greedy",
               init_size=100, batch_size=100, n_rounds=n_rounds, topk=50,
               run_dir=Path(tempfile.mkdtemp()), seed=42, **kw)
print("ready")

---
## Exercise 1: `LoudExplorer`, a subclass that prints the 3 best each round

In [ ]:
class LoudExplorer(rx.MVEExplorer):
    def _checkpoint(self, rnd, record):
        best3 = sorted(self.labeled_scores.items(), key=lambda kv: kv[1])[:3]   # lowest score = best
        print(f"  [loud] after round {rnd}: best so far", [(str(s), round(float(v), 2)) for s, v in best3])
        super()._checkpoint(rnd, record)          # still write iter_N/ like the parent does

ex = explorer(LoudExplorer, surrogate_epochs=10)
history = ex.run()
print("files written:", sorted(p.name for p in ex.run_dir.iterdir()))

**Answer.** `LoudExplorer` inherits everything from `MVEExplorer` and replaces only `_checkpoint`. The loop in `run()` calls `self._checkpoint(...)` after every round, and because `self` is a `LoudExplorer`, Python finds the new version first. That's how overriding works. Calling `super()._checkpoint(rnd, record)` at the end runs the parent's version too, so `iter_1/` and `iter_2/` still get written (check the file list). Leave that line out and the printing still works, but the checkpoints silently disappear, so `--resume` would no longer work. When you override a method, think about whether the parent's job still needs doing.

---
## Exercise 2: `RandomSurrogate`

In [ ]:
class RandomSurrogate:
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)
    def fit(self, X, y, **kwargs):              # **kwargs: accept epochs=/batch= and ignore them
        pass
    def predict(self, X):
        return self.rng.normal(size=len(X)), np.zeros(len(X))

finals = []
for seed in range(5):
    ex = explorer(n_rounds=4)
    ex.model.surrogate = RandomSurrogate(seed)   # the swap from Part 11
    with contextlib.redirect_stdout(io.StringIO()):
        h = ex.run()
    finals.append(h[-1]["topk_recall"])
print("RandomSurrogate final recall, 5 seeds:", finals, " mean", np.mean(finals))
print("expected for random search: labeled / pool =", 500 / N)

**Answer.** Around 0.14 over 5 seeds, against an expected 0.167 for random search (500 of 3,000 molecules docked). That's within the luck you'd expect with only 50 target molecules: each seed's result is a multiple of 2%, and they range from 10% to 20%. The point is the comparison. The real surrogate in the lesson found 80–100% with the same budget, so a model that guesses randomly is exactly as good as docking at random. This is also a useful **sanity check** to run on any new pipeline: plug in a random model, and if it scores far above random, something is leaking the answer.

---
## Exercise 3: break the contract (`predict` returns only `mu`)

In [ ]:
class OnlyMu:
    def fit(self, X, y, **kwargs):
        pass
    def predict(self, X):
        return np.zeros(len(X))                  # contract says (mu, sigma); this returns only mu

ex = explorer(n_rounds=1)
ex.model.surrogate = OnlyMu()
try:
    with contextlib.redirect_stdout(io.StringIO()):
        ex.run()
except Exception:
    traceback.print_exc(limit=-3, file=sys.stdout)   # just the last 3 steps of the traceback

**Answer.** It fails inside the wrapper, at `mvemodels.py`, in `get_means_and_vars`: `mu, sigma = self.surrogate.predict(self._get_X(xs))`, with `ValueError: too many values to unpack (expected 2)`. Python tried to split the returned array into exactly two things and found 2,800 values instead. Notice that the error doesn't mention "contract" or "sigma". You have to read the traceback from the bottom and recognise that the line expects two return values. That's why the class guide calls `fit`/`predict` a contract by convention: nothing checks it until something downstream breaks.

---
## Exercise 4: trace the LT-All surrogate

In [ ]:
ex = explorer(n_rounds=1, surrogate_type="ltall", surrogate_kwargs={"epochs": 5})
_ = trace(ex.run, max_depth=3)

**Answer.** LT-All takes the **normal** branch of `_chunked_get_means_and_vars`: the trace shows `EmbeddingMVEModel.get_means_and_vars` → `LTAllSurrogate.predict`, not `_chunked_predict_components` and `combine`. The reason is in Part 7's table: `LTAllSurrogate` has no `combine` method, so `hasattr(surrogate, "combine")` is false. LT-All scores every molecule independently (no ranks), so it's safe to predict chunk by chunk, which is why it never had the per-chunk ranking bug from notebook 8. Also visible in the trace: `_train_one x5`, its 5 independently trained networks.

---
## Exercise 5: what replaces `_chunked_get_means_and_vars` in `ParallelMVEExplorer.run`?

We print only the lines of `ParallelMVEExplorer.run` that talk to the workers: the
markers, the `.done` files, and loading the workers' predictions.

In [ ]:
lines = inspect.getsource(rx.ParallelMVEExplorer.run).splitlines()
start = inspect.getsourcelines(rx.ParallelMVEExplorer.run)[1]
for i, line in enumerate(lines):
    if any(word in line for word in ["marker", ".done", "_wait_for_shard", "np.load", "surrogate.pt", "torch.save", "concatenate"]):
        print(f"{start + i:5d}  {line.strip()[:110]}")

**Answer.** In `ParallelMVEExplorer.run`, the single call `mu, var = _chunked_get_means_and_vars(self.model, pool_smi)` is replaced by a hand-off to the workers. The orchestrator saves the trained surrogate (`torch.save(... round_{r}_surrogate.pt)`), announces it (`round_{r}_ready.marker`), then for each shard waits for its `.done` file (`_wait_for_shard`) and loads that shard's `..._mu.npy` and `..._var.npy`. Finally `np.concatenate` glues the shards back into one `mu` and one `var` for the whole pool. From there on, acquisition is the same as in `MVEExplorer`. After the last round it writes `STOP.marker` so the workers exit. The line numbers printed above are live, so they stay correct even when the file changes.